# ML, round 2
Testing ML performance when data of neighboring pixels is also provided for classification.

## Imports

In [1]:
import random

import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, accuracy_score, classification_report
from sklearn.base import BaseEstimator, ClassifierMixin

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

from hist_via_cluster.load_dataset import (
    load_pixel_dataframe_with_neighborhood,
    load_fluorescence,
)


def get_project_root():
    root = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
    return root.parent if root.name == "models" else root



## Config

In [2]:
# Reproducibility
RANDOM_STATE = 42

# Help function to set seeds inside the training function
def set_global_seed(seed: int):
    """Set seeds for Python, NumPy and PyTorch for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    # Optional but recommended for determinism:
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    # For PyTorch to error on non-deterministic ops:
    # torch.use_deterministic_algorithms(True)

set_global_seed(RANDOM_STATE)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DEVICE

device(type='cpu')

## 0. Loading the Dataset

In [3]:
PROJECT_ROOT = get_project_root()
DATA_ROOT = PROJECT_ROOT / "data"

NEIGHBORHOOD_SIZE = 5  # e.g., 5x5 patches
df = load_pixel_dataframe_with_neighborhood(
    directory=str(DATA_ROOT),
    neighborhood_size=NEIGHBORHOOD_SIZE,
)
df.head()
# df["label"].value_counts()

,diet,mouse,take,row,col,Ca_-2_-2,Ca_-2_-1,Ca_-2_+0,Ca_-2_+1,Ca_-2_+2,...,Zn_+1_-1,Zn_+1_+0,Zn_+1_+1,Zn_+1_+2,Zn_+2_-2,Zn_+2_-1,Zn_+2_+0,Zn_+2_+1,Zn_+2_+2,label
0,2,20,0,2,2,84181600.0,106057000.0,133411000.0,140305000.0,311744000.0,...,5309360.0,0.0,0.0,188244.0,0.0,3620580.0,0.0,0.0,5360040.0,NaN
1,2,20,0,2,3,106057000.0,133411000.0,140305000.0,311744000.0,465376000.0,...,0.0,0.0,188244.0,26774200.0,3620580.0,0.0,0.0,5360040.0,29172500.0,NaN
2,2,20,0,2,4,133411000.0,140305000.0,311744000.0,465376000.0,488834000.0,...,0.0,188244.0,26774200.0,32889900.0,0.0,0.0,5360040.0,29172500.0,26333000.0,NaN
3,2,20,0,2,5,140305000.0,311744000.0,465376000.0,488834000.0,424265000.0,...,188244.0,26774200.0,32889900.0,57623700.0,0.0,5360040.0,29172500.0,26333000.0,59287600.0,NaN
4,2,20,0,2,6,311744000.0,465376000.0,488834000.0,424265000.0,369933000.0,...,26774200.0,32889900.0,57623700.0,35557400.0,5360040.0,29172500.0,26333000.0,59287600.0,44690400.0,NaN


## 1. Task definition
Para esta primera aproximación, hacemos coarse graining en las etiquetas. Voy a tirar los artificios (label actual 5), y redefino
1. necrótico (que en estos casos no hay)
2. tumoral (labels 2, 3, y 4)
3. no tumoral (labels 6, 7 y 9)
4. no muestra (labels 8 y 10)

Los datos no etiquetados (0 o NaN) se quedan en `NaN`. El hecho de agrupar las etiquetas tumorales tiene sentido más allá de nuestro objetivo, porque estos puntos se encuentran relativamente cerca en el espacio de fluorescencia. Recordemos que las distribuciones de las dos clases (2 y 3) se solapan significativamente en el espacio de la componente dada por LDA, y la etiqueta 4 fue filtrada en el corte de la curación de los datos.

## 2. Coarse-graining + label construction

In [4]:
# ============================================================
#  SECTION 2 — Data loading + coarse-graining + label construction
# ============================================================

# -------------------------
# Load data
# -------------------------
PROJECT_ROOT = get_project_root()
DATA_ROOT = PROJECT_ROOT / "data"

NEIGHBORHOOD_SIZE = 5  # choose e.g. 3, 5, 7, ...
df = load_pixel_dataframe_with_neighborhood(
    directory=str(DATA_ROOT),
    neighborhood_size=NEIGHBORHOOD_SIZE,
)

# -------------------------
# Coarse-graining (YOUR MAPPING)
# -------------------------

def coarse_graining_map(label):
    if pd.isna(label) or label == 0:
        return np.nan
    elif label == 1:             # necrotic tissue (not present in your dataset)
        return 1
    elif label in [2, 3, 4]:     # tumoral tissue
        return 2
    elif label in [6, 7, 9]:     # non-tumoral tissue
        return 3
    elif label in [8, 10]:       # empty sample (not sample)
        return 4
    else:
        return np.nan

coarse_graining_labels = {
    np.nan: 'no label',
    1: 'necrótico',
    2: 'tumoral',
    3: 'no tumoral',
    4: 'no muestra'
}

df["new_label"] = df["label"].apply(coarse_graining_map)

# -------------------------
# Identify the categories that appear in this dataset
# -------------------------

present = sorted(df["new_label"].dropna().unique().tolist())
print("Categories present (new_label):", present)
# Should print: [2, 3, 4]

# -------------------------
# Construct binary labels for the two-stage classifier
# -------------------------

# Stage 1: sample vs not-sample
#
# sample:   new_label ∈ {2 (tumor), 3 (non-tumor)}
# not-sample: new_label == 4

df["y_stage1"] = df["new_label"].isin([2, 3]).astype(int)

# Stage 2: tumor vs non-tumor (only meaningful if y_stage1 == 1)
#
# tumor:     new_label == 2
# non-tumor: new_label == 3
# not-sample: irrelevant / undefined, will be masked out when training Stage 2

df["y_stage2"] = (df["new_label"] == 2).astype(int)

# -------------------------
# Feature column identification
# -------------------------

META_COLS = ["diet", "mouse", "take", "row", "col", "label",
             "new_label", "y_stage1", "y_stage2"]

feature_cols = [c for c in df.columns if c not in META_COLS]

print("Number of feature columns:", len(feature_cols))

# -------------------------
# Retrieve element order and construct mapping (elem, ro, co) → feature col
# -------------------------

ds_dict = load_fluorescence(str(DATA_ROOT), as_dict=True)
element_order = ds_dict.element_order
n_elements = len(element_order)

k = (NEIGHBORHOOD_SIZE - 1) // 2

def parse_feature_name(name: str):
    """Parse feature name 'Ca_+1_-1' → (elem_str, row_offset, col_offset)."""
    elem, ro, co = name.split("_")
    return elem, int(ro), int(co)

# Build mapping
col_index_map = {}
for idx, name in enumerate(feature_cols):
    elem, ro, co = parse_feature_name(name)
    elem_idx = element_order.index(elem)
    col_index_map[(elem_idx, ro, co)] = idx

# Sanity check: confirm all expected offsets exist
offsets = range(-k, k + 1)
for elem_idx in range(n_elements):
    for ro in offsets:
        for co in offsets:
            assert (elem_idx, ro, co) in col_index_map, f"Missing {(elem_idx, ro, co)}"

print("Finished Section 2: df with new_label, y_stage1, y_stage2, feature_cols, col_index_map.")


Categories present (new_label): [2.0, 3.0, 4.0]
Number of feature columns: 200
Finished Section 2: df with new_label, y_stage1, y_stage2, feature_cols, col_index_map.


## 3. PyTorch Dataset for patches

In [5]:
class PatchDataset(Dataset):
    """
    Dataset yielding (patch, label) pairs for binary classification.

    Patch is shaped as (C, H, W), where C = number of elements,
    H = W = NEIGHBORHOOD_SIZE.

    The input DataFrame is expected to have:
    - feature columns in `feature_cols`
    - binary labels in `y_column` (0/1)
    """

    def __init__(
        self,
        df: pd.DataFrame,
        feature_cols,
        col_index_map,
        y_column: str,
        mask: np.ndarray,
        n_elements: int,
        neighborhood_size: int,
        dtype=torch.float32,
    ):
        self.df = df.reset_index(drop=True)
        self.feature_cols = feature_cols
        self.col_index_map = col_index_map
        self.y_column = y_column
        self.indices = np.nonzero(mask)[0]
        self.n_elements = n_elements
        self.k = (neighborhood_size - 1) // 2
        self.size = neighborhood_size
        self.dtype = dtype

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        row_idx = self.indices[idx]
        row = self.df.iloc[row_idx]

        # Extract features as numpy array
        x_flat = row[self.feature_cols].to_numpy(dtype=np.float32)
        # Reconstruct patch: shape (C, H, W)
        patch = np.zeros((self.n_elements, self.size, self.size), dtype=np.float32)

        for (elem_idx, ro, co), col_idx in self.col_index_map.items():
            # Map (ro,co) in [-k,k] into indices [0, size-1]
            i = self.k - ro     # row offset: +ro -> up (decreasing row index)
            j = self.k + co     # col offset: +co -> right (increasing col index)
            patch[elem_idx, i, j] = x_flat[col_idx]

        y = int(row[self.y_column])

        x_tensor = torch.from_numpy(patch).to(self.dtype)
        y_tensor = torch.tensor(y, dtype=torch.float32)

        return x_tensor, y_tensor


## 4. Train–val-calib-test split for each stage

We want separate splits for stage 1 and 2, with stratification:
- Stage 1: sample vs not-sample, uses all labeled pixels;
- Stage 2: tumor vs non-tumor, only for sample pixels.

In [6]:
# ============================================================
#  CONSISTENT TRAIN / VAL / CALIB / TEST SPLITS
#  + DATASETS AND DATALOADERS FOR BOTH STAGES
# ============================================================

from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader

# -------------------------
# 1. Define global indices and valid pixels
# -------------------------

N = len(df)
all_idx = np.arange(N)

# Keep only pixels with a well-defined coarse-grained label {2,3,4}
valid_mask = df["new_label"].isin([2, 3, 4]).to_numpy()
idx_valid = all_idx[valid_mask]

y_cg_valid = df.loc[idx_valid, "new_label"].to_numpy()  # stratification labels

# -------------------------
# 2. Split into train / val / calib / test
#
# Scheme (approx):
#   1) 80% train+val+calib, 20% test
#   2) From 80%, 80% train+val, 20% calib
#   3) From train+val, 75% train, 25% val
#
# Final proportions ≈
#   train ≈ 0.48
#   val   ≈ 0.16
#   calib ≈ 0.16
#   test  ≈ 0.20
# -------------------------

# Step 1: hold-out test set
idx_train_val_calib, idx_test = train_test_split(
    idx_valid,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y_cg_valid,
)

y_cg_tvc = df.loc[idx_train_val_calib, "new_label"].to_numpy()

# Step 2: calibration set from remaining (train+val+calib)
idx_train_val, idx_calib = train_test_split(
    idx_train_val_calib,
    test_size=0.20,  # 20% of 80% -> 16% overall
    random_state=RANDOM_STATE,
    stratify=y_cg_tvc,
)

y_cg_tv = df.loc[idx_train_val, "new_label"].to_numpy()

# Step 3: training vs validation
idx_train, idx_val = train_test_split(
    idx_train_val,
    test_size=0.25,  # 25% of 64% -> 16% overall
    random_state=RANDOM_STATE,
    stratify=y_cg_tv,
)

# -------------------------
# 3. Build boolean masks for each split
# -------------------------
mask_train = np.zeros(N, dtype=bool)
mask_val = np.zeros(N, dtype=bool)
mask_calib = np.zeros(N, dtype=bool)
mask_test = np.zeros(N, dtype=bool)

mask_train[idx_train] = True
mask_val[idx_val] = True
mask_calib[idx_calib] = True
mask_test[idx_test] = True

print("Split sizes (all pixels with new_label ∈ {2,3,4}):")
print("  train :", mask_train.sum())
print("  val   :", mask_val.sum())
print("  calib :", mask_calib.sum())
print("  test  :", mask_test.sum())

# -------------------------
# 4. Stage-specific masks
#
# Stage 1 (sample vs not-sample):
#   uses ALL pixels in each split (new_label ∈ {2,3,4}).
#
# Stage 2 (tumor vs non-tumor):
#   only pixels with new_label ∈ {2,3} (i.e., y_stage1 == 1).
# -------------------------

is_sample = df["new_label"].isin([2, 3]).to_numpy()  # same as (df["y_stage1"] == 1)

# Stage 1 masks
mask_train_s1 = mask_train & valid_mask
mask_val_s1   = mask_val   & valid_mask
mask_calib_s1 = mask_calib & valid_mask
mask_test_s1  = mask_test  & valid_mask

# Stage 2 masks (only sample pixels)
mask_train_s2 = mask_train & is_sample
mask_val_s2   = mask_val   & is_sample
mask_calib_s2 = mask_calib & is_sample
mask_test_s2  = mask_test  & is_sample

print("\nStage 1 (all valid pixels) sizes:")
print("  train :", mask_train_s1.sum())
print("  val   :", mask_val_s1.sum())
print("  calib :", mask_calib_s1.sum())
print("  test  :", mask_test_s1.sum())

print("\nStage 2 (only sample pixels) sizes:")
print("  train :", mask_train_s2.sum())
print("  val   :", mask_val_s2.sum())
print("  calib :", mask_calib_s2.sum())
print("  test  :", mask_test_s2.sum())

# -------------------------
# 5. Build PatchDataset instances for both stages
#    (PatchDataset defined earlier in Section 3)
# -------------------------

BATCH_SIZE = 2048  # adjust as needed

# --- Stage 1: y_stage1 (sample vs no-sample) ---

train_ds1 = PatchDataset(
    df=df,
    feature_cols=feature_cols,
    col_index_map=col_index_map,
    y_column="y_stage1",
    mask=mask_train_s1,
    n_elements=n_elements,
    neighborhood_size=NEIGHBORHOOD_SIZE,
)

val_ds1 = PatchDataset(
    df=df,
    feature_cols=feature_cols,
    col_index_map=col_index_map,
    y_column="y_stage1",
    mask=mask_val_s1,
    n_elements=n_elements,
    neighborhood_size=NEIGHBORHOOD_SIZE,
)

calib_ds1 = PatchDataset(
    df=df,
    feature_cols=feature_cols,
    col_index_map=col_index_map,
    y_column="y_stage1",
    mask=mask_calib_s1,
    n_elements=n_elements,
    neighborhood_size=NEIGHBORHOOD_SIZE,
)

test_ds1 = PatchDataset(
    df=df,
    feature_cols=feature_cols,
    col_index_map=col_index_map,
    y_column="y_stage1",
    mask=mask_test_s1,
    n_elements=n_elements,
    neighborhood_size=NEIGHBORHOOD_SIZE,
)

train_loader1 = DataLoader(train_ds1, batch_size=BATCH_SIZE, shuffle=True)
val_loader1   = DataLoader(val_ds1,   batch_size=BATCH_SIZE, shuffle=False)
calib_loader1 = DataLoader(calib_ds1, batch_size=BATCH_SIZE, shuffle=False)
test_loader1  = DataLoader(test_ds1,  batch_size=BATCH_SIZE, shuffle=False)

# --- Stage 2: y_stage2 (tumor vs non-tumor, only on sample pixels) ---

train_ds2 = PatchDataset(
    df=df,
    feature_cols=feature_cols,
    col_index_map=col_index_map,
    y_column="y_stage2",
    mask=mask_train_s2,
    n_elements=n_elements,
    neighborhood_size=NEIGHBORHOOD_SIZE,
)

val_ds2 = PatchDataset(
    df=df,
    feature_cols=feature_cols,
    col_index_map=col_index_map,
    y_column="y_stage2",
    mask=mask_val_s2,
    n_elements=n_elements,
    neighborhood_size=NEIGHBORHOOD_SIZE,
)

calib_ds2 = PatchDataset(
    df=df,
    feature_cols=feature_cols,
    col_index_map=col_index_map,
    y_column="y_stage2",
    mask=mask_calib_s2,
    n_elements=n_elements,
    neighborhood_size=NEIGHBORHOOD_SIZE,
)

test_ds2 = PatchDataset(
    df=df,
    feature_cols=feature_cols,
    col_index_map=col_index_map,
    y_column="y_stage2",
    mask=mask_test_s2,
    n_elements=n_elements,
    neighborhood_size=NEIGHBORHOOD_SIZE,
)

train_loader2 = DataLoader(train_ds2, batch_size=BATCH_SIZE, shuffle=True)
val_loader2   = DataLoader(val_ds2,   batch_size=BATCH_SIZE, shuffle=False)
calib_loader2 = DataLoader(calib_ds2, batch_size=BATCH_SIZE, shuffle=False)
test_loader2  = DataLoader(test_ds2,  batch_size=BATCH_SIZE, shuffle=False)

print("\nDataLoaders are ready:")
print("  Stage 1: train/val/calib/test")
print("  Stage 2: train/val/calib/test")


Split sizes (all pixels with new_label ∈ {2,3,4}):
  train : 5586
  val   : 1862
  calib : 1862
  test  : 2328

Stage 1 (all valid pixels) sizes:
  train : 5586
  val   : 1862
  calib : 1862
  test  : 2328

Stage 2 (only sample pixels) sizes:
  train : 4246
  val   : 1416
  calib : 1416
  test  : 1770

DataLoaders are ready:
  Stage 1: train/val/calib/test
  Stage 2: train/val/calib/test


## 5. MLP model with rotation-invariant loss

In [7]:
# Model
class MLPBinary(nn.Module):
    """
    Simple MLP for binary classification on flattened patches.

    Input: (B, C, H, W)
    Output: logits shape (B, 1)
    """

    def __init__(
        self,
        n_channels: int,
        patch_size: int,
        hidden_dims=(128, 64),
        dropout=0.1,
    ):
        super().__init__()
        input_dim = n_channels * patch_size * patch_size

        layers = []
        in_dim = input_dim
        for h in hidden_dims:
            layers.append(nn.Linear(in_dim, h))
            layers.append(nn.ReLU())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            in_dim = h
        layers.append(nn.Linear(in_dim, 1))  # logits

        self.net = nn.Sequential(*layers)

    def forward(self, x):
        # Use reshape instead of view (safe for non-contiguous tensors)
        x_flat = x.reshape(x.size(0), -1)
        logits = self.net(x_flat)
        return logits  # (B, 1)


In [8]:
# Rotation-invariant BCE loss
bce_loss = nn.BCEWithLogitsLoss()

def rotation_invariant_bce_loss(model, x, y, criterion=bce_loss, n_rotations=4):
    """
    Average BCE loss over n_rotations = 4 (0, 90, 180, 270 degrees).

    x: (B, C, H, W)
    y: (B,) with values in {0,1}
    """
    total_loss = 0.0
    for k in range(n_rotations):
        x_rot = torch.rot90(x, k=k, dims=(-2, -1))
        logits = model(x_rot).squeeze(1)  # (B,)
        total_loss = total_loss + criterion(logits, y)
    return total_loss / n_rotations


## 6. Training Loops

Helper for training one model:

In [9]:
def train_binary_mlp(
    train_dataset,
    val_dataset,
    n_channels,
    patch_size,
    hidden_dims=(128, 64),
    lr=1e-3,
    weight_decay=1e-4,
    max_epochs=20,
    rotation_invariant=True,
    batch_size=2048,
    seed=RANDOM_STATE,
    device=DEVICE,
):
    """
    Deterministic training:
    - Re-seeds Python/NumPy/PyTorch at the beginning.
    - Creates DataLoaders with a fixed torch.Generator for shuffling.
    """

    # 1) Reset all RNGs so re-running the cell repeats exactly
    set_global_seed(seed)

    # 2) Build fresh DataLoaders with a fixed generator for shuffle
    g = torch.Generator()
    g.manual_seed(seed)

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        generator=g,
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
    )

    # 3) Initialize model and optimizer
    model = MLPBinary(
        n_channels=n_channels,
        patch_size=patch_size,
        hidden_dims=hidden_dims,
    ).to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

    for epoch in range(1, max_epochs + 1):
        model.train()
        train_losses = []

        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)

            optimizer.zero_grad()

            if rotation_invariant:
                loss = rotation_invariant_bce_loss(model, xb, yb)
            else:
                logits = model(xb).squeeze(1)
                loss = bce_loss(logits, yb)

            loss.backward()
            optimizer.step()

            train_losses.append(loss.item())

        # Validation
        model.eval()
        val_losses = []
        all_y = []
        all_scores = []

        with torch.no_grad():
            for xb, yb in val_loader:
                xb = xb.to(device)
                yb = yb.to(device)

                logits = model(xb).squeeze(1)
                loss = bce_loss(logits, yb)
                val_losses.append(loss.item())

                all_y.append(yb.cpu().numpy())
                all_scores.append(torch.sigmoid(logits).cpu().numpy())

        all_y = np.concatenate(all_y)
        all_scores = np.concatenate(all_scores).ravel()
        val_auc = roc_auc_score(all_y, all_scores)

        print(
            f"[Epoch {epoch:03d}] "
            f"train_loss={np.mean(train_losses):.4f} "
            f"val_loss={np.mean(val_losses):.4f} "
            f"val_auc={val_auc:.4f}"
        )

    return model

We train stage 1 and stage 2 models:

In [21]:
model_stage1 = train_binary_mlp(
    train_dataset=train_ds1,
    val_dataset=val_ds1,
    n_channels=n_elements,
    patch_size=NEIGHBORHOOD_SIZE,
    max_epochs=13,
    batch_size=2048,
    seed=RANDOM_STATE,  # optional, but explicit
)

model_stage2 = train_binary_mlp(
    train_dataset=train_ds2,
    val_dataset=val_ds2,
    n_channels=n_elements,
    patch_size=NEIGHBORHOOD_SIZE,
    max_epochs=9,
    batch_size=2048,
    seed=RANDOM_STATE,  # same seed, new model, same data order
)


[Epoch 001] train_loss=8137123.2500 val_loss=4730813.0000 val_auc=0.5112
[Epoch 002] train_loss=6058173.1667 val_loss=6222615.5000 val_auc=0.5370
[Epoch 003] train_loss=5595010.5000 val_loss=2427927.0000 val_auc=0.7349
[Epoch 004] train_loss=3351464.4167 val_loss=998848.1250 val_auc=0.8050
[Epoch 005] train_loss=2559157.1667 val_loss=1836814.2500 val_auc=0.7401
[Epoch 006] train_loss=2414944.9167 val_loss=1588147.8750 val_auc=0.7378
[Epoch 007] train_loss=1785652.0417 val_loss=483171.0625 val_auc=0.8317
[Epoch 008] train_loss=1594279.2917 val_loss=403051.0625 val_auc=0.8290
[Epoch 009] train_loss=1179214.5208 val_loss=767234.6875 val_auc=0.7854
[Epoch 010] train_loss=1084984.0625 val_loss=301475.3750 val_auc=0.8692
[Epoch 011] train_loss=1025188.0417 val_loss=507536.9688 val_auc=0.8231
[Epoch 012] train_loss=916759.6250 val_loss=445454.5625 val_auc=0.8359
[Epoch 013] train_loss=781692.4792 val_loss=265030.6562 val_auc=0.8718
[Epoch 001] train_loss=10034540.1667 val_loss=6938750.0000 va

Note to self: training on 25 epochs for first and second stage models yielded the following losses and accuracies. Note that the best validation accuracies are:
- 13 for the stage 1 model
- 9 for the stage 2 model
Thus, we'll set those values as max epochs. (This is hyperparameter tuning — early-stopping actually — by hand, so by doing this I've already used the information of the validation set.)

```
[Epoch 001] train_loss=8137123.2500 val_loss=4730813.0000 val_auc=0.5112
[Epoch 002] train_loss=6058173.1667 val_loss=6222615.5000 val_auc=0.5370
[Epoch 003] train_loss=5595010.5000 val_loss=2427927.0000 val_auc=0.7349
[Epoch 004] train_loss=3351464.4167 val_loss=998848.1250 val_auc=0.8050
[Epoch 005] train_loss=2559157.1667 val_loss=1836814.2500 val_auc=0.7401
[Epoch 006] train_loss=2414944.9167 val_loss=1588147.8750 val_auc=0.7378
[Epoch 007] train_loss=1785652.0417 val_loss=483171.0625 val_auc=0.8317
[Epoch 008] train_loss=1594279.2917 val_loss=403051.0625 val_auc=0.8290
[Epoch 009] train_loss=1179214.5208 val_loss=767234.6875 val_auc=0.7854
[Epoch 010] train_loss=1084984.0625 val_loss=301475.3750 val_auc=0.8692
[Epoch 011] train_loss=1025188.0417 val_loss=507536.9688 val_auc=0.8231
[Epoch 012] train_loss=916759.6250 val_loss=445454.5625 val_auc=0.8359

[Epoch 013] train_loss=781692.4792 val_loss=265030.6562 val_auc=0.8718

[Epoch 014] train_loss=664332.5625 val_loss=359625.3438 val_auc=0.8398
[Epoch 015] train_loss=633749.1667 val_loss=332297.6250 val_auc=0.8474
[Epoch 016] train_loss=560784.9792 val_loss=262391.5938 val_auc=0.8648
[Epoch 017] train_loss=523795.5104 val_loss=285629.5625 val_auc=0.8481
[Epoch 018] train_loss=469484.7812 val_loss=266860.4375 val_auc=0.8397
[Epoch 019] train_loss=430088.6250 val_loss=184235.3438 val_auc=0.8641
[Epoch 020] train_loss=400574.9062 val_loss=155164.7656 val_auc=0.8653
[Epoch 021] train_loss=369428.6771 val_loss=133000.5625 val_auc=0.8629
[Epoch 022] train_loss=336565.8333 val_loss=119004.2031 val_auc=0.8613
[Epoch 023] train_loss=314773.2500 val_loss=112455.2812 val_auc=0.8621
[Epoch 024] train_loss=294573.4167 val_loss=95430.9609 val_auc=0.8634
[Epoch 025] train_loss=267540.8646 val_loss=90949.0859 val_auc=0.8425



[Epoch 001] train_loss=10034540.1667 val_loss=6938750.0000 val_auc=0.5000
[Epoch 002] train_loss=8017065.5000 val_loss=7582647.0000 val_auc=0.5000
[Epoch 003] train_loss=5437239.6667 val_loss=1465038.5000 val_auc=0.5242
[Epoch 004] train_loss=4320608.5833 val_loss=1843837.1250 val_auc=0.5738
[Epoch 005] train_loss=3078300.0833 val_loss=2871947.5000 val_auc=0.5516
[Epoch 006] train_loss=2760663.9583 val_loss=816451.0000 val_auc=0.6207
[Epoch 007] train_loss=2551053.8333 val_loss=498329.6875 val_auc=0.6725
[Epoch 008] train_loss=1789572.3750 val_loss=1232414.6250 val_auc=0.5783

[Epoch 009] train_loss=1738144.6250 val_loss=267126.3125 val_auc=0.8064

[Epoch 010] train_loss=1505991.2917 val_loss=326917.5000 val_auc=0.7373
[Epoch 011] train_loss=1131922.3542 val_loss=814865.6875 val_auc=0.6036
[Epoch 012] train_loss=1324045.8333 val_loss=336495.7812 val_auc=0.7154
[Epoch 013] train_loss=1006031.9167 val_loss=276080.6250 val_auc=0.7449
[Epoch 014] train_loss=936693.4583 val_loss=499268.0625 val_auc=0.6360
[Epoch 015] train_loss=847342.3750 val_loss=348166.6875 val_auc=0.6887
[Epoch 016] train_loss=733963.4375 val_loss=298412.4375 val_auc=0.7101
[Epoch 017] train_loss=699140.4167 val_loss=388891.2500 val_auc=0.6391
[Epoch 018] train_loss=619238.6667 val_loss=247471.3438 val_auc=0.7215
[Epoch 019] train_loss=555749.8333 val_loss=267720.5938 val_auc=0.6992
[Epoch 020] train_loss=531350.8958 val_loss=213170.9531 val_auc=0.7760
[Epoch 021] train_loss=472503.2708 val_loss=280421.3125 val_auc=0.7233
[Epoch 022] train_loss=490614.0208 val_loss=195901.3281 val_auc=0.7652
[Epoch 023] train_loss=497921.9167 val_loss=181002.2969 val_auc=0.7907
[Epoch 024] train_loss=406766.8854 val_loss=180407.8438 val_auc=0.8047
[Epoch 025] train_loss=367921.0208 val_loss=196106.7188 val_auc=0.6846
```

## 7. Calibration: Collect logits + Fit Platt Scaler

In [22]:
from sklearn.linear_model import LogisticRegression

def collect_logits_and_labels(model, data_loader, device=DEVICE):
    model.eval()
    all_logits = []
    all_y = []
    with torch.no_grad():
        for xb, yb in data_loader:
            xb = xb.to(device)
            yb = yb.to(device)

            logits = model(xb).squeeze(1)
            all_logits.append(logits.cpu().numpy())
            all_y.append(yb.cpu().numpy())

    return np.concatenate(all_logits), np.concatenate(all_y)


def fit_platt_scaler(logits, y):
    clf = LogisticRegression(
        solver="lbfgs",
        max_iter=500,
        random_state=RANDOM_STATE,
    )
    clf.fit(logits.reshape(-1, 1), y)
    return clf


# Stage 1
logits_calib1, y_calib1 = collect_logits_and_labels(model_stage1, calib_loader1)
calibrator1 = fit_platt_scaler(logits_calib1, y_calib1)

# Stage 2
logits_calib2, y_calib2 = collect_logits_and_labels(model_stage2, calib_loader2)
calibrator2 = fit_platt_scaler(logits_calib2, y_calib2)


Calibrated probability function

In [23]:
def calibrated_proba(model, calibrator, x, device=DEVICE):
    model.eval()
    with torch.no_grad():
        x = x.to(device)
        logits = model(x).squeeze(1)
        logits_np = logits.cpu().numpy().reshape(-1, 1)
    return calibrator.predict_proba(logits_np)[:, 1]


## 8. Bayesian Decision Rule (Cost-Sensitive)

In [30]:
class BayesianDecisionRule:
    """
    Bayes-optimal decision rule with asymmetric misclassification costs:
    C10 = cost(false positive), C01 = cost(false negative)

    threshold = C10 / (C10 + C01)
    """

    def __init__(self, C10=1.0, C01=1.0):
        self.C10 = float(C10)
        self.C01 = float(C01)
        self.threshold = self.C10 / (self.C10 + self.C01)

    def predict(self, p):
        p = np.asarray(p)
        return (p >= self.threshold).astype(int)

    def __repr__(self):
        return f"BayesianDecisionRule(C10={self.C10}, C01={self.C01}, threshold={self.threshold:.3f})"


# Example:
rule_stage1 = BayesianDecisionRule(C10=1.0, C01=1.0)  # equal costs for sample vs not-sample
# rule_stage2 = BayesianDecisionRule(C10=1.0, C01=2.0)  # penalize missing tumors
rule_stage2 = BayesianDecisionRule(C10=1.0, C01=1.0)


## 9. Two-Stage Predictor with 3-Way Output

In [31]:
# Patch-building helper
def build_patch_tensor_for_rows(
    df_subset,
    feature_cols,
    col_index_map,
    n_elements,
    neighborhood_size,
    dtype=torch.float32,
):
    k = (neighborhood_size - 1) // 2
    size = neighborhood_size
    N = len(df_subset)

    patches = np.zeros((N, n_elements, size, size), dtype=np.float32)
    X = df_subset[feature_cols].to_numpy(dtype=np.float32)

    for i in range(N):
        x_flat = X[i]
        patch = patches[i]
        for (elem_idx, ro, co), col_idx in col_index_map.items():
            r = k - ro
            c = k + co
            patch[elem_idx, r, c] = x_flat[col_idx]

    return torch.from_numpy(patches).to(dtype)


In [32]:
def two_stage_predict_df(
    df_subset,
    feature_cols,
    col_index_map,
    model_stage1,
    calibrator1,
    rule_stage1,
    model_stage2,
    calibrator2,
    rule_stage2,
    n_elements,
    neighborhood_size,
    device=DEVICE
):
    x_tensor = build_patch_tensor_for_rows(
        df_subset,
        feature_cols,
        col_index_map,
        n_elements=n_elements,
        neighborhood_size=neighborhood_size,
    )

    # Stage 1: sample vs not-sample
    p_sample = calibrated_proba(model_stage1, calibrator1, x_tensor, device=device)
    yhat_sample = rule_stage1.predict(p_sample)

    # Stage 2: tumor vs non-tumor only for predicted sample
    p_tumor = np.zeros_like(p_sample)
    yhat_tumor = np.zeros_like(yhat_sample)

    is_sample = (yhat_sample == 1)
    if is_sample.any():
        x_sample = x_tensor[is_sample]
        p_tumor_sample = calibrated_proba(model_stage2, calibrator2, x_sample, device=device)
        yhat_tumor_sample = rule_stage2.predict(p_tumor_sample)
        p_tumor[is_sample] = p_tumor_sample
        yhat_tumor[is_sample] = yhat_tumor_sample

    # Final 3-class label
    #   0 = not sample
    #   1 = non-tumoral
    #   2 = tumoral
    final = np.zeros_like(yhat_sample)
    final[(is_sample) & (yhat_tumor == 0)] = 1
    final[(is_sample) & (yhat_tumor == 1)] = 2

    return final, p_sample, p_tumor


## 10. Final Evaluation on the Test Set

In [33]:
# We build the true labels (consistent with the mapping)
true_label = df["new_label"].to_numpy()

# Map into {0,1,2}
# 4 -> 0 (not sample)
# 3 -> 1 (non-tumoral)
# 2 -> 2 (tumoral)
true_final = np.zeros_like(true_label, dtype=int)
true_final[true_label == 3] = 1
true_final[true_label == 2] = 2

# Evaluate only on the test split
df_test = df.loc[mask_test]
true_test_final = true_final[mask_test]


In [34]:
# We get the predictions
final_pred, p_sample_test, p_tumor_test = two_stage_predict_df(
    df_subset=df_test,
    feature_cols=feature_cols,
    col_index_map=col_index_map,
    model_stage1=model_stage1,
    calibrator1=calibrator1,
    rule_stage1=rule_stage1,
    model_stage2=model_stage2,
    calibrator2=calibrator2,
    rule_stage2=rule_stage2,
    n_elements=n_elements,
    neighborhood_size=NEIGHBORHOOD_SIZE,
)


In [35]:
from sklearn.metrics import classification_report, confusion_matrix

print("=== FINAL TEST SET REPORT (3-way classifier) ===")
print(classification_report(true_test_final, final_pred, digits=3))

print("=== CONFUSION MATRIX (rows = true, cols = predicted) ===")
print(confusion_matrix(true_test_final, final_pred))


=== FINAL TEST SET REPORT (3-way classifier) ===
              precision    recall  f1-score   support

           0      0.827     0.841     0.834       558
           1      0.926     0.228     0.366       276
           2      0.862     0.977     0.916      1494

    accuracy                          0.856      2328
   macro avg      0.872     0.682     0.705      2328
weighted avg      0.862     0.856     0.831      2328

=== CONFUSION MATRIX (rows = true, cols = predicted) ===
[[ 469    1   88]
 [  68   63  145]
 [  30    4 1460]]
